# Manual LLM and Simple Agent Tutorial with no libraries

In [1]:
# -------------------------
# Standard Library
# -------------------------
import time # part of agent telemetry ~ how long a prompt took, tokens/sec
from pathlib import Path # instead of writing '"/Users/catherine/Documents/models/model.gguf"'

# -------------------------
# Llama.cpp Python Binding
# Path: Python -> llama_cpp.Llama -> llama.cpp -> GGUF model
# -------------------------
from llama_cpp import Llama

## Notes

```(agentic_ai) catherineordun@Catherines-Air Agentic Interactive Tutorial % ls ~/Documents/llama.cpp/models
Llama-3.2-3B-Instruct-Q4_K_M.gguf
ggml-vocab-aquila.gguf
ggml-vocab-baichuan.gguf
ggml-vocab-bert-bge.gguf
ggml-vocab-bert-bge.gguf.inp
ggml-vocab-bert-bge.gguf.out
ggml-vocab-command-r.gguf
ggml-vocab-command-r.gguf.inp
ggml-vocab-command-r.gguf.out
ggml-vocab-deepseek-coder.gguf
ggml-vocab-deepseek-coder.gguf.inp
ggml-vocab-deepseek-coder.gguf.out
ggml-vocab-deepseek-llm.gguf
ggml-vocab-deepseek-llm.gguf.inp
ggml-vocab-deepseek-llm.gguf.out
ggml-vocab-falcon.gguf
ggml-vocab-falcon.gguf.inp
ggml-vocab-falcon.gguf.out
ggml-vocab-gpt-2.gguf
ggml-vocab-gpt-2.gguf.inp
ggml-vocab-gpt-2.gguf.out
ggml-vocab-gpt-neox.gguf
ggml-vocab-llama-bpe.gguf
ggml-vocab-llama-bpe.gguf.inp
ggml-vocab-llama-bpe.gguf.out
ggml-vocab-llama-spm.gguf
ggml-vocab-llama-spm.gguf.inp
ggml-vocab-llama-spm.gguf.out
ggml-vocab-mpt.gguf
ggml-vocab-mpt.gguf.inp
ggml-vocab-mpt.gguf.out
ggml-vocab-nomic-bert-moe.gguf
ggml-vocab-phi-3.gguf
ggml-vocab-phi-3.gguf.inp
ggml-vocab-phi-3.gguf.out
ggml-vocab-qwen2.gguf
ggml-vocab-qwen2.gguf.inp
ggml-vocab-qwen2.gguf.out
ggml-vocab-refact.gguf
ggml-vocab-refact.gguf.inp
ggml-vocab-refact.gguf.out
ggml-vocab-starcoder.gguf
ggml-vocab-starcoder.gguf.inp```

These are the vocabulary files that llama.cpp downloads. 


## What the `ggml-vocab` means: 
Everything named like this:

`ggml-vocab-*.gguf`

is a vocabulary/tokenizer test fixture that ships with llama.cpp. These files are used by llama.cpp’s tests and examples to verify that different tokenizer formats work correctly, for example:

`ggml-vocab-llama-bpe.gguf`: Llama BPE tokenizer fixture
`ggml-vocab-phi-3.gguf`: Phi-3 tokenizer fixture

```
ggml-vocab-llama-bpe.gguf
ggml-vocab-llama-bpe.gguf.inp
ggml-vocab-llama-bpe.gguf.out
```

The .inp and .out files are paired test data:

`.inp`: sample input text/tokens for tokenizer tests

`.out`: expected tokenizer output

In [2]:
model_path = (Path.home()
/ "Documents"
/ "llama.cpp"
/ "models"
/ "Llama-3.2-3B-Instruct-Q4_K_M.gguf"
)

print(model_path)
print(model_path.exists())

/Users/catherineordun/Documents/llama.cpp/models/Llama-3.2-3B-Instruct-Q4_K_M.gguf
True


## n_gpu_layers

`n_gpu_layers = 0` Put it all on CPU

`n_gpu_layers=10` Put the first 10 on GPU, all else CPU

`n_gpu_layers=20` Put the first 20 on GPU, all else CPU

`n_gpu_layers=-1` Offload as many layers as possible on GPU like 'ALL'

## unified memory 
Remember, M3 has unified memory, meaning CPU & GPU share it, there is no PCIe Bus that requires you to copy numpy arrays back and forth on CPU to GPU, no `.cuda()` or `to_cuda()` transform. 


```
# CUDA/NVIDIA
import torch
import numpy as np

np_array = np.array([[1,2,3], [2,3,4], [3,4,5]])
cpu_tensor = torch.from_numpy(np_array)
gpu_tensor = cpu_tensor.to("cuda") <- Copies data to VRAM and the speed of copy is limited by PCIEbus speed
# Also, now two copies live in RAM: the cpu_tensor and the gpu_tensor
```

```
# On MAC
np_array = np.array([[1,2,3], [2,3,4], [3,4,5]])
cpu_tensor = torch.from_numpy(np_array)
^^ all that is the same

mac_tensor = cpu_tensor.to("mps") <- Data stay in pooled/unified memory, just uses a pointer

```




In [3]:
# Ask Python to open the GGUF file
# Then load into memory
# Then ask questions from it

llm = Llama(model_path=str(model_path), 
            n_ctx=2048, # max context window - what it remembers in short-term memory
            n_gpu_layers=-1, # use Apple's Metal GPU
            verbose=False)
print("Model loaded succesfully!")

# memory: 512 small, 2048 good, 8192 good GP, 32000 long conversations

Model loaded succesfully!


## GGUF Files

GGUF files contain: 1) Metadata of the model and 2) Tensors, the numeric weights

Control tokens are special tokens for the machine to mark: 

```
beginning of text
end of text
start of header
end of header
end of turn
```

`EOG` = End of Generation 
`Token-to-piece cache`: this is the tokenization mapping like `token ID -> text fragment` like `9906 -> "Hello"`

Llama has a small lookup cache that occupies 0.8 MB memory. Like an old-school NLTK dictionary.

`BPE` = Byte-Pair Encoding; Llama doesn't assign a token ID for every word. Often times a fragment of text is split up into multiple fragments: `unbelievable` => `un` + `believ` + `able`; It is a compromise between characters and words (words are efficient packaging of characters). So at inference time, Llama BEP will break up prompt, tokenizer -> token IDs. 


## Inference

In [4]:
response = llm.create_chat_completion(
    messages = [
        {
            "role": "user",
            "content": "In one sentence, tell me what an AI agent is."
        }
    ]
)

In [5]:
response #is a Python dictionary 

{'id': 'chatcmpl-d63c7846-f229-4fc8-98c1-16b4b784bbfd',
 'object': 'chat.completion',
 'created': 1784504340,
 'model': '/Users/catherineordun/Documents/llama.cpp/models/Llama-3.2-3B-Instruct-Q4_K_M.gguf',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'An AI agent is a software program that uses artificial intelligence (AI) and machine learning (ML) to interact with and make decisions in a specific environment or system, often with the goal of achieving a particular objective or outcome.'},
   'logprobs': None,
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 47, 'completion_tokens': 45, 'total_tokens': 92}}

## About the tokens

Prompt: "In one sentence, tell me what an AI agent is."

Before Llama receives the prompt, it passes through a TOKENIZER first. That TOKENIZER (much like my NLTK dict/GloVE example) maps token IDs to words or fragments. But becuase it was wrapped in `create_chat_completion()` it passed it to Llama and used the special tokens ("control tokens") to wrap it like this: 

```
<|begin_of_text|>

<|start_header_id|>user<|end_header_id|>

In one sentence, tell me what an AI agent is.

<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>

```

And as a result, each of those DO COUNT. 

Total prompt tokens: 
- actual words
- punctutation
- spaces (BPE encodes leading spaces)
- special chat tokens
- role markers
- end-of-turn markers

## Tokenizer

Like NLTK or scikit-learn, the tokenizer is run before the neural network. As a result, this can be used to estimate the number of tokens **before** it's wrapped in Llama. This means: 

- estimate cost
- estimate latency
- measure context usage
- debug prompts

In [6]:
prompt = "In one sentence, tell me what an AI agent is."
tokens = llm.tokenize(prompt.encode("utf-8"))
print(tokens) #128000 is a start token indicator

[128000, 644, 832, 11914, 11, 3371, 757, 1148, 459, 15592, 8479, 374, 13]


In [7]:
print(f"Number of tokens:{len(tokens)}")

Number of tokens:13


**The application consumes most of your tokens, not the user**

Suppose your agent has:

a 500-token system prompt,
20 conversation turns,
retrieved documents,
tool outputs,

The user's question might only be 20 tokens, but the actual prompt sent to the model could be 2,000+ tokens after everything is assembled.

In [8]:
# convert every token back to text 

for token in tokens: 
    piece = llm.detokenize([token]).decode("utf-8", errors="replace")
    print(f"{token:>7} -> {repr(piece)}")

 128000 -> ''
    644 -> 'In'
    832 -> ' one'
  11914 -> ' sentence'
     11 -> ','
   3371 -> ' tell'
    757 -> ' me'
   1148 -> ' what'
    459 -> ' an'
  15592 -> ' AI'
   8479 -> ' agent'
    374 -> ' is'
     13 -> '.'


In [9]:
type(tokens)

list

In [10]:
type(token)

int

In [11]:
examples = [
    "AI",
    " AI",
    "agent",
    " agent",
    "ChatGPT",
    " ChatGPT",
    "😊",
    "🚀",
    "The quick brown fox jumps over the lazy dog."
]

for text in examples: 
    ids = llm.tokenize(text.encode("utf-8"))
    print(f"{repr(text):35} -> {len(ids)}:2 tokens -> {ids}")

'AI'                                -> 2:2 tokens -> [128000, 15836]
' AI'                               -> 2:2 tokens -> [128000, 15592]
'agent'                             -> 2:2 tokens -> [128000, 8252]
' agent'                            -> 2:2 tokens -> [128000, 8479]
'ChatGPT'                           -> 4:2 tokens -> [128000, 16047, 38, 2898]
' ChatGPT'                          -> 4:2 tokens -> [128000, 13149, 38, 2898]
'😊'                                 -> 3:2 tokens -> [128000, 76460, 232]
'🚀'                                 -> 4:2 tokens -> [128000, 9468, 248, 222]
'The quick brown fox jumps over the lazy dog.' -> 11:2 tokens -> [128000, 791, 4062, 14198, 39935, 35308, 927, 279, 16053, 5679, 13]


## Embeddings -> Assign Meaning

**Just a look up table** so that we can do math

To find, for example, `token ID = 8479`, model looks it up in row 8479 of the embedding matrix, which is multidimensional.

Llama for 3.2 3B has 3072 fp values. 
This means that: 
- if there are 13 token IDs
- it is 3072 dimensions each

So that token IDs: 
644, 832, 11914... 

**Each** becomes [3072 floats], [3072 floats], [3072 floats] ...

Only **THEN** DOES ATTENTION START. 




Prompt: "In one sentence, tell me what an AI agent is."

Tokenizer: 128000
644
832
11914
...

Now it becomes a **matrix**: 
13 × 3072

```
┌──────────────────────────────┐
│ Token 1 │ • • • • • • • • • │
│ Token 2 │ • • • • • • • • • │
│ Token 3 │ • • • • • • • • • │
│ Token 4 │ • • • • • • • • • │
│   ...   │ • • • • • • • • • │
└──────────────────────────────┘
```

- Embedding layer outputs: 3072 values
- Attention layer (comes after embedding) accepts: 3072 values
- Every MLP/FF layer returns: 3072 values
- The final hidden state: 3072 values
- Width is constant


```
3072
   │
Attention
   │
3072
   │
Feed Forward
   │
3072
   │
Attention
   │
3072
```


In [12]:
print(llm.n_embd())

3072


## Transformer

So far we have gone through: 

```
English sentence
        │
        ▼
Tokenizer
        │
        ▼
Token IDs
        │
        ▼
Embedding Lookup
        │
        ▼
3072-dimensional vectors
        │
        ▼
❓ Self-Attention
```

In [13]:
import torch

print(torch.__version__)
print(torch.backends.mps.is_available())

2.13.0
True


## Tiny Experiment

3 tokens -> small 4D embedding -> Q,K,V projections -> attention scores -> context-aware vectors

In [14]:
tokens = ["dog", "chased", "ball"]

# this is our pretend 4D embedding
x = torch.tensor([
    [1.0, 0.0, 1.0, 0.0],  # dog
    [0.0, 1.0, 1.0, 0.0],  # chased
    [0.0, 1.0, 0.0, 1.0],  # ball
])

print(x)
print("Shape:", x.shape)

tensor([[1., 0., 1., 0.],
        [0., 1., 1., 0.],
        [0., 1., 0., 1.]])
Shape: torch.Size([3, 4])


### Projection Matrices

Model (Llama) learns a matrix that transforms embeddings into query vectors, or key vectors, or value vectors.

In [15]:
import torch.nn as nn

d_model = 4 # per our toy 4D embedding

# linear transformation tensor layer
W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)

print("W_q shape:", W_q.weight.shape)
print("W_k shape:", W_k.weight.shape)
print("W_v shape:", W_v.weight.shape)

W_q shape: torch.Size([4, 4])
W_k shape: torch.Size([4, 4])
W_v shape: torch.Size([4, 4])


In [16]:
#[3,4] x [4,4] = [3,4]

# pass x (the toy embedding) into the FF/linear layer networks
q = W_q(x) # Logic: Linear transformation (W_q) of the embedding (x) outputs the query
k = W_k(x)
v = W_v(x)

# now we have our query, keys, and values
print("q shape:", q.shape) # this means [3 tokens, 4 dimensions]
print("k shape:", k.shape)
print("v shape:", v.shape)

q shape: torch.Size([3, 4])
k shape: torch.Size([3, 4])
v shape: torch.Size([3, 4])


### Transpose
Required for matrix multiplication to calculate raw attention

Pytorch dimensions: 

```
positive index:     0   1
shape:             [3,  4]
negative index:    -2  -1
```

In [17]:
k_transposed = k.transpose(-2, -1) # swap second-to-last, with last, better this way for when the tensors grow in dimensionality

print("k shape:      ", k.shape)
print("k transposed shape:", k_transposed.shape)

k shape:       torch.Size([3, 4])
k transposed shape: torch.Size([4, 3])


### Raw Attention

`q x k^T` = raw scores

`[3,4] @ [4,3]` 
- where each row is a query token: "What am I being asked?"
- and each column is the key: "What do I have?"
And eventually, value, answers: "What should I return?"


```
             key: dog   chased   ball
query: dog
query: chased
query: ball
```


In [18]:
q_k = torch.matmul(q, k_transposed)

print("q shape:  ", q.shape)
print("kᵀ shape: ", k_transposed.shape)
print("q_k shape:", q_k.shape)

print("\nRaw attention scores:")
print(q_k)

q shape:   torch.Size([3, 4])
kᵀ shape:  torch.Size([4, 3])
q_k shape: torch.Size([3, 3])

Raw attention scores:
tensor([[-0.2033, -0.1891,  0.0533],
        [-0.3959, -0.0980, -0.3603],
        [-0.0738, -0.0777, -0.2999]], grad_fn=<MmBackward0>)


In [19]:
print(q_k[0,2])

tensor(0.0533, grad_fn=<SelectBackward0>)


### Scale

scaled scores = QK^T / sqrt(d_k) # d_k: dimensions of k matrix

Why? As the vectors become wider, their dot products grow in magnitude. Very large scores can cause softmax to **become extremely sharp**, with nearly all probability assigned to one position. Scaling keeps the numbers in a more manageable range.

In [ ]:
d_k = k.size(-1)
print(d_k)

In [ ]:
k.shape # [3 tokens, 4 dimensions]

In [ ]:
import math

d_k = k.size(-1)
scale = math.sqrt(d_k)

scaled_scores = q_k/scale

print("d_k:", d_k)
print("sqrt(d_k):", scale)

print("\nRaw scores:")
print(q_k)

print("\nScaled scores:")
print(scaled_scores)

### Build the Causal Mask

Don't let the token look at future tokens. Mask them out. 

```
dog     → dog
chased  → dog, chased
ball    → dog, chased, ball
```

So when you look at the QK matrix, you have to block out some tokens so it doesn't see future patterns: 

```
             key
             dog    chased   ball
query dog     keep   <block>   <block>
query chased  keep    keep      <block>
query ball    keep    keep      keep

where <block> = True
```


In [ ]:
mask = torch.triu(
    torch.ones(
        scaled_scores.shape,
        dtype=torch.bool
    ),
    diagonal=1 # each token attends to itself
)

print("Mask shape:", mask.shape)
print(mask)

Now replace Trues and Falses: 

```
True  → replace with -∞
False → keep the original score
```

B/c softmax turns negative infinity into probability 0. 

In [ ]:
masked_scores = scaled_scores.masked_fill(
    mask,
    float("-inf")
)

print("Scaled scores:")
print(scaled_scores)

print("\nMasked scores:")
print(masked_scores)

### Softmax
convert each row of masked probabilities into probabilities.

For the given query token, how much attention should go to each key? 

In [ ]:
import torch.nn.functional as F

attention_weights = F.softmax(masked_scores, dim=-1)
print(attention_weights)
print(attention_weights.sum(dim=-1)) # last dim is the keys [3,3]

### Compute Value Vectors (*V) - Blends information

```
attention_weights shape = [3 queries, 3 tokens]
v shape                 = [3 tokens, 4 value dimensions]

[3, 3] @ [3, 4] → [3, 4]

output[2]
=
0.2803 × v[0]
+
0.3292 × v[1]
+
0.3905 × v[2]
```

In [ ]:
attention_output = torch.matmul(attention_weights, v)



print("Attention weights shape:", attention_weights.shape)
print("V shape:", v.shape)
print("Output shape:", attention_output.shape)

print("\nValue vectors:")
print(v)

print("\nAttention output:")
print(attention_output)

In [ ]:
attention_weights

In [ ]:
v

### Verify one row

proves that `torch.matmul(attention_weights, v)` is just performing a weighted sum. 

In [ ]:
ball_output_manual = (
    attention_weights[2, 0] * v[0]
    + attention_weights[2, 1] * v[1]
    + attention_weights[2, 2] * v[2]
)

print("Manual ball output:")
print(ball_output_manual)

print("\nMatrix multiplication ball output:")
print(attention_output[2])

print("\nAre they equal?")
print(torch.allclose(
    ball_output_manual,
    attention_output[2]
))

# Minimal Agent

Building a core loop: 

`request` -> `agent f(x)` -> `LLM call` -> `result`

## Agent Request

Dictionary is the agent's state and request envelope. 

Note: This is not from the LLM. The agent is a **Python program** that decides when and how to use the LLM.

In [16]:
#vmm is my own "vistamorph more" algorithm, it can be anything like 'agent'

# Python dictionary
# This is a mapping, like a contract, where task == str('summarize') we could have
# called it 'banana' and it would still map to this input. 
vmm_request = {
    "task": "summarize", # What to do
    "input": ( # What to process
        "Agentic AI systems combine language models "
        "with tools, memory, and control logic."
    ),
    "user_id": "catherine", # Whio requested it
}

print(vmm_request)
print("\nRequest type:", type(vmm_request))

# In the future you can add:
# "tools", "memory", "constraints", "request_id", "max_steps"

{'task': 'summarize', 'input': 'Agentic AI systems combine language models with tools, memory, and control logic.', 'user_id': 'catherine'}

Request type: <class 'dict'>


## Build first agent function

In [ ]:
def run_agent(request):
    print("--Agent started--")

    print(f"Task:     {request['task']}") # task
    print(f"User:     {request['user_id']}") #user_id
    print(f"Input:    {request['input']}") #input

    print("\nDecision:")
    print("-> I should call the language model.")

    print("---Agent finished---")

In [ ]:
run_agent(vmm_request)

The LLM is only a resource!

```
User -> Python agent -> Decision -> LLM -> Decision -> Tool? -> Memory? -> LLM? -> Answer

```

## Call Llama (LLM)

Agent has to do two things: 

1) Decide what to do.
2) Ask the LLM to do it. 

<span style="color:red; font-weight:bold">An Agent</span>

In [ ]:
# hard code the decision as an example
# task == summarize -> build a prompt -> call Llama

def run_agent(request):
    print("---Agent started---")

    task = request["task"]
    text = request["input"]

    print("Task:", task)

    if task=="summarize": 
        prompt = (
            "Summarize the following in one sentence:\n\n"
            f"{text}"
        )

        print("\nPrompt sent to Llama:")
        print("-" * 40)
        print(prompt)
        print("-" * 40)

        # this is when we call the LLM
        response = llm.create_chat_completion(
            messages = [
                {
                    "role": "user", 
                    "content": prompt,
                }
            ],
            temperature=0.2,
        )
    
        answer = response["choices"][0]["message"]["content"]
    
        print("\nModel response:")
        print(answer)
    
        return answer

    else:
        print("Unknown task.")



In [ ]:
run_agent(vmm_request)

# Simple Tool

We could ask the LLM: "What is 234 x 567?" But it can make mistakes. 
Instead, we could ask the agent to use a calculator tool. 

Here, the Agent choose whether to call the tool or the LLM. 
```
             Python Agent
           ┌───────┴────────┐
           ▼                ▼
     Calculator Tool      LLM
           │                │
           └───────┬────────┘
                   ▼
                Response
                
```

<span style="color:red; font-weight:bold">A tool</span>

In [4]:
# Remember this tool is just a Python function
def calculator(a,b):
    print("Calculator tool called.")
    return a*b

In [ ]:
answer = calculator(234,567)
print("Result:", answer)

In [7]:
math_request = {
    "task":"multiply",
    "a":234,
    "b":567,
    "user_id":"catherine",
}

In [ ]:
# now the agent will choose to summarize or multiply 

def run_agent(request):
    print("---Agent started---")

    task=request["task"]
    print("Task", task)

    if task == "multiply":
        result = calculator(request["a"], request["b"])
        print("Agent result:", result)
        return result

    print("Unknown task")

In [ ]:
result = run_agent(math_request)
print(result)

## Add the LLM route back in

<span style="color:red; font-weight:bold">Tool routing</span>

In [5]:
def run_agent(request):
    print("---Agent started---")

    task=request["task"]
    print("Task:", task)

    if task=="multiply":
        # use the calculator tool
        result=calculator(request["a"], request["b"])
        print("Agent route: calculator")
        print("Agent result:", result)
        return result

    elif task=="summarize":
        prompt=("Summarize the following in one sentence:\n\n"
               f"{request['input']}")
        print("Agent route: LLM")

        # then call the LLM
        response=llm.create_chat_completion(
            messages=[
                {
                "role":"user",
                "content": prompt,
                }
            ],
            temperature=0.2
        )
        print("Response:\n\n"
              f"{response}")
        result = response["choices"][0]["message"]["content"]

        print("Agent result:", result)
        return result

    else:
        print("Unkown task.")
        return None

    

In [8]:
run_agent(math_request)

---Agent started---
Task: multiply
Calculator tool called.
Agent route: calculator
Agent result: 132678


132678

In [17]:
summary_result=run_agent(vmm_request)
print(summary_result)

---Agent started---
Task: summarize
Agent route: LLM
Response:

{'id': 'chatcmpl-3f22db22-41fe-466a-98d5-e8018f33dacf', 'object': 'chat.completion', 'created': 1784505812, 'model': '/Users/catherineordun/Documents/llama.cpp/models/Llama-3.2-3B-Instruct-Q4_K_M.gguf', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Agentic AI systems integrate language models with additional components such as tools, memory, and control logic to enable autonomous decision-making and action.'}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 60, 'completion_tokens': 27, 'total_tokens': 87}}
Agent result: Agentic AI systems integrate language models with additional components such as tools, memory, and control logic to enable autonomous decision-making and action.
Agentic AI systems integrate language models with additional components such as tools, memory, and control logic to enable autonomous decision-making and action.


In [ ]:
response["choices"][0]

## Decision Prompt

From a fixed router to a LLM-directed agent.

**A set of instructions** <- 

Now, in a modern agentic system, the LLM will determine that the calculator is the right tool to use. It is not hardcoded in via Python as `"task":"multiply"`

In [9]:
# "task":"multiply" is now replaced by...

tool_request = {
    "input":"What is 234 multipled by 567?",
    "user_id":"catherine"
}
print(tool_request)

{'input': 'What is 234 multipled by 567?', 'user_id': 'catherine'}


The LLM needs to return a **decision** not an answer.

```
user question -> LLM decides: "user calcuator with a=234 and b=567"

```

<span style="color:red; font-weight:bold">LLM Planning</span>

In [10]:
#builds and prints instructions that will be sent to the LLM
# under "Return only valid JSON", you could ask it to return anything
# that's text: JSON, Python code, SQL, HTML, XML, Markdown, tool calls, API requests

decision_prompt=f"""

You are a tool router.

Available tool: 

calculator(a,b)
- Multiplies two numbers.

User request:
{tool_request["input"]}

Return only valid JSON in this exact format: 

{{
    "tool":"calculator",
    "arguments": {{
        "a":0,
        "b":0}}
}}


"""

print(decision_prompt)



You are a tool router.

Available tool: 

calculator(a,b)
- Multiplies two numbers.

User request:
What is 234 multipled by 567?

Return only valid JSON in this exact format: 

{
    "tool":"calculator",
    "arguments": {
        "a":0,
        "b":0}
}





## Ask Llama for the routing decision

In [11]:
response = llm.create_chat_completion(
    messages=[
        {
            "role": "user",
            "content": decision_prompt, # calls the decision_prompt
        }
    ],
    temperature=0,
    max_tokens=100,
)

decision = response["choices"][0]["message"]["content"]

print(decision)

{
    "tool":"calculator",
    "arguments": {
        "a":234,
        "b":567}
}


<span style="color:red; font-weight:bold">JSON Parsing</span>

In [12]:
import json 

tool_call=json.loads(decision)

print(tool_call)
type(tool_call)
tool_call["arguments"]

{'tool': 'calculator', 'arguments': {'a': 234, 'b': 567}}


{'a': 234, 'b': 567}

Now Python will do this: 

```
read tool name -> read arguments -> call calculator() -> receive result
```

<span style="color:red; font-weight:bold">Tool execution</span>

In [13]:
tool_name = tool_call["tool"]
arguments = tool_call["arguments"]

if tool_name=="calculator":
    tool_result = calculator(
        arguments["a"],
        arguments["b"]
    )
else:
    tool_result = None
    print("Unknown tool", tool_name)

print(tool_result)

Calculator tool called.
132678


## Call Llama in a second loop

In the first loop we wrote a `decision_prompt` to return JSON, then loaded the JSON into as the response, mapping it to the `tool_call`. 

```
             User
               │
               ▼
        LLM decides tool
               │
               ▼
      Python executes tool
               │
               ▼
        Tool returns result # All of the above just done
               │
               ▼
        LLM writes answer # Now we're here
               │
               ▼
             User

```

The first prompt asked: "What tool should I use?"

The second prompt asks: "Now that you have the tool result, talk to the user."

These are completely different jobs. The LLM is acting in two different roles:

1. Planner / Router
2. Communicator

Many production agent systems use the same model for both roles, while others use different models (for example, a small fast model for routing and a larger model for the final response).

In [14]:
final_prompt = f"""
The calculator returned the following result:

{tool_result}

Write a friendly answer to the user.
"""

print(final_prompt)


The calculator returned the following result:

132678

Write a friendly answer to the user.



**The actual second call** that communicates back to the user. 

Our tiny agent already called it twice:

(1) To reason about what action to take.
(2) To communicate the result back to the user.

Some production agents call an LLM 10–50 times to solve a single request.

In [19]:
response = llm.create_chat_completion(
    messages=[
        {
            "role": "user",
            "content": final_prompt,
        }
    ],
    temperature=0.2,
    max_tokens=100,
)

final_answer = response["choices"][0]["message"]["content"]

print(final_answer)

It looks like your calculator has done its magic! The result you've got is indeed 132,678. Would you like to know what calculation led to this number?


## Tool Registry

In [20]:
tools = {
    "calculator": calculator
}
print(tools) # storing a reference to the function

{'calculator': <function calculator at 0x10af90dc0>}


In [21]:
# this dictionary contains a function
print(tools["calculator"])
print(type(tools["calculator"]))

<function calculator at 0x10af90dc0>
<class 'function'>


In [22]:
my_tool = tools["calculator"]
result = my_tool(10,20)
result

Calculator tool called.


200

## Langchain

In our decision_prompt, it looked like this: 

```
decision_prompt=f"""

You are a tool router.

Available tool: 

calculator(a,b)
- Multiplies two numbers.

User request:
{tool_request["input"]}

Return only valid JSON in this exact format: 

{{
    "tool":"calculator",
    "arguments": {{
        "a":0,
        "b":0}}
}}


"""
```

If we had multiple tools, like weather(), gmail(), calendar(), etc. we would have to write them into this `decision_prompt` with all those tools which is inefficient. 

What Langchain does is use a **docstring** as the description and automatically inserts into the prompt. LangChain builds the available tools. 

How? 

- Uses `@tools` decorator instead of manally creating a `tools` dictionary. 
- `PromptTemplate` instead of building prompts with f-strings
- AgentExecutor `create_agent()` which ties together the planning loop we just built

### Go to second notebook